In [93]:
import pandas as pd
import numpy as np

data = pd.read_csv("../data/sites_data.csv")
data.head()

,site_id,act_timeline,expected_recruit_rate,rec_perf_ratio,op_reliab,startup_cost,month_op_cost
0,Site_001,36,3.43,0.71,0.64,10186,2873
1,Site_002,84,4.40,0.76,0.64,7481,2000
2,Site_003,75,2.54,0.69,0.64,7562,2000
3,Site_004,60,1.62,0.88,0.84,11765,2991
4,Site_005,60,1.56,1.05,0.65,7554,2000


In [94]:
print("Total rows:", len(data))
print("Unique sites:", data["site_id"].nunique())
print("Duplicate site IDs:", data["site_id"].duplicated().sum())

Total rows: 500
Unique sites: 500
Duplicate site IDs: 0


In [95]:
indications_by_area = {
    "Infectious Diseases": [
        "Malaria",
        "HIV-1",
        "Tuberculosis"
    ],
    "Cardiovascular": [
        "Hypertension",
        "Heart failure",
        "Coronary artery disease"
    ],
    "Oncology": [
        "Cervical cancer",
        "Breast cancer",
        "Prostate cancer"
    ],
    "Hematology": [
        "Sickle cell disease",
        "Iron-deficiency anaemia",
        "Haemophilia"
    ],
    "Respiratory Diseases": [
        "Pneumonia",
        "COPD",
        "Asthma"
    ]
}

In [96]:
countries = [
    "Egypt",
    "Algeria",
    "Morocco",
    "Nigeria",
    "South Africa",
    "Uganda",
    "Kenya",
    "Ghana",
    "Tanzania",
    "Ethiopia"
]

In [97]:
area_preferences = {
    "Egypt": {
        "Infectious Diseases": 2,
        "Cardiovascular": 3,
        "Oncology": 3,
        "Hematology": 2,
        "Respiratory Diseases": 2
    },
    "Algeria": {
        "Infectious Diseases": 2,
        "Cardiovascular": 3,
        "Oncology": 3,
        "Hematology": 2,
        "Respiratory Diseases": 2
    },
    "Morocco": {
        "Infectious Diseases": 2,
        "Cardiovascular": 3,
        "Oncology": 3,
        "Hematology": 2,
        "Respiratory Diseases": 2
    },
    "Nigeria": {
        "Infectious Diseases": 3,
        "Cardiovascular": 2,
        "Oncology": 2,
        "Hematology": 2,
        "Respiratory Diseases": 2
    },
    "South Africa": {
        "Infectious Diseases": 3,
        "Cardiovascular": 3,
        "Oncology": 3,
        "Hematology": 2,
        "Respiratory Diseases": 3
    },
    "Uganda": {
        "Infectious Diseases": 3,
        "Cardiovascular": 2,
        "Oncology": 2,
        "Hematology": 2,
        "Respiratory Diseases": 3
    },
    "Kenya": {
        "Infectious Diseases": 3,
        "Cardiovascular": 2,
        "Oncology": 3,
        "Hematology": 2,
        "Respiratory Diseases": 2
    },
    "Ghana": {
        "Infectious Diseases": 3,
        "Cardiovascular": 2,
        "Oncology": 2,
        "Hematology": 2,
        "Respiratory Diseases": 2
    },
    "Tanzania": {
        "Infectious Diseases": 3,
        "Cardiovascular": 2,
        "Oncology": 2,
        "Hematology": 2,
        "Respiratory Diseases": 3
    },
    "Ethiopia": {
        "Infectious Diseases": 3,
        "Cardiovascular": 2,
        "Oncology": 2,
        "Hematology": 2,
        "Respiratory Diseases": 2
    }
}

In [98]:
#control point
areas = list(indications_by_area.keys())

for country in countries:
    assert country in area_preferences
    assert set(area_preferences[country].keys()) == set(areas)
    assert all(
        weight > 0
        for weight in area_preferences[country].values()
    )

print("All countries have valid preferences")

All countries have valid preferences


In [99]:
rng = np.random.default_rng(42)

data["country"] = rng.choice(
    countries,
    size=len(data),
    replace=True
)

data[["site_id", "country"]].head()

,site_id,country
0,Site_001,Egypt
1,Site_002,Ghana
2,Site_003,Kenya
3,Site_004,South Africa
4,Site_005,South Africa


In [100]:
country_counts = (
    data["country"]
    .value_counts()
    .reindex(countries, fill_value=0)
)

country_counts

country
Egypt           47
Algeria         58
Morocco         44
Nigeria         43
South Africa    64
Uganda          46
Kenya           47
Ghana           59
Tanzania        43
Ethiopia        49
Name: count, dtype: int64

In [101]:
n_sites = len(data)

n_basic = round(n_sites * 0.40)
n_broad = round(n_sites * 0.40)
n_advanced = n_sites - n_basic - n_broad

area_counts = (
    [2] * n_basic
    + [3] * n_broad
    + rng.choice([4, 5], size=n_advanced).tolist()
)

rng.shuffle(area_counts)

data["n_therapeutic_areas"] = area_counts

data["n_therapeutic_areas"].value_counts().sort_index()

n_therapeutic_areas
2    200
3    200
4     48
5     52
Name: count, dtype: int64

In [102]:
site_areas = {}

for _, row in data.iterrows():
    country = row["country"]
    n_areas = row["n_therapeutic_areas"]

    available_areas = list(indications_by_area.keys())

    weights = np.array([
        area_preferences[country][area]
        for area in available_areas
    ], dtype=float)

    probabilities = weights / weights.sum()

    selected_areas = rng.choice(
        available_areas,
        size=n_areas,
        replace=False,
        p=probabilities
    )

    site_areas[row["site_id"]] = selected_areas.tolist()

In [103]:
assigned_area_counts = pd.Series({
    site_id: len(areas)
    for site_id, areas in site_areas.items()
})

print("Total sites:", len(site_areas))
print(
    "Sites with correct n° of areas:",
    (assigned_area_counts == data.set_index("site_id")
     ["n_therapeutic_areas"]).sum()
)
print(
    "Any duplicated areas within a site:",
    any(
        len(areas) != len(set(areas))
        for areas in site_areas.values()
    )
)

Total sites: 500
Sites with correct n° of areas: 500
Any duplicated areas within a site: False


In [104]:
site_indications = {}

for site_id, assigned_areas in site_areas.items():
    indications = []

    for area in assigned_areas:
        available_indications = indications_by_area[area]

        n_indications = rng.integers(
            1,
            min(3, len(available_indications)) + 1
        )

        selected_indications = rng.choice(
            available_indications,
            size=n_indications,
            replace=False
        )

        for indication in selected_indications:
            indications.append({
                "site_id": site_id,
                "therapeutic_area": area,
                "indication": indication
            })

    site_indications[site_id] = indications

In [105]:
clinical_profiles = pd.DataFrame(
    [
        record
        for indications in site_indications.values()
        for record in indications
    ]
)

clinical_profiles.head(10)

,site_id,therapeutic_area,indication
0,Site_001,Respiratory Diseases,Asthma
1,Site_001,Respiratory Diseases,Pneumonia
2,Site_001,Infectious Diseases,Malaria
3,Site_001,Infectious Diseases,HIV-1
4,Site_001,Infectious Diseases,Tuberculosis
5,Site_001,Oncology,Cervical cancer
6,Site_001,Oncology,Prostate cancer
7,Site_002,Hematology,Haemophilia
8,Site_002,Infectious Diseases,HIV-1
9,Site_002,Cardiovascular,Heart failure


In [106]:
#controlpoint
print("Total clinical profiles:", len(clinical_profiles))
print("Duplicate site-area-indication combinations:",
    clinical_profiles.duplicated(
        subset=["site_id", "therapeutic_area", "indication"]
    ).sum()
)
print("Sites without clinical profiles:",
    len(set(data["site_id"]) - set(clinical_profiles["site_id"]))
)

Total clinical profiles: 2858
Duplicate site-area-indication combinations: 0
Sites without clinical profiles: 0


In [107]:
clinical_profiles = clinical_profiles.merge(
    data[["site_id", "country"]],
    on="site_id",
    how="left",
    validate="many_to_one"
)

clinical_profiles = clinical_profiles[
    ["site_id", "country", "therapeutic_area", "indication"]
]

clinical_profiles.head(10)

,site_id,country,therapeutic_area,indication
0,Site_001,Egypt,Respiratory Diseases,Asthma
1,Site_001,Egypt,Respiratory Diseases,Pneumonia
2,Site_001,Egypt,Infectious Diseases,Malaria
3,Site_001,Egypt,Infectious Diseases,HIV-1
4,Site_001,Egypt,Infectious Diseases,Tuberculosis
5,Site_001,Egypt,Oncology,Cervical cancer
6,Site_001,Egypt,Oncology,Prostate cancer
7,Site_002,Ghana,Hematology,Haemophilia
8,Site_002,Ghana,Infectious Diseases,HIV-1
9,Site_002,Ghana,Cardiovascular,Heart failure


In [108]:
#cp
area_distribution = (
    clinical_profiles
    .groupby(["country", "therapeutic_area"])["site_id"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(index=countries, columns=areas, fill_value=0)
)

area_distribution

therapeutic_area,Infectious Diseases,Cardiovascular,Oncology,Hematology,Respiratory Diseases
country,,,,,
Egypt,30,29,26,19,27
Algeria,30,39,40,29,28
Morocco,26,28,23,29,28
Nigeria,34,18,19,27,26
South Africa,38,41,40,38,38
Uganda,26,27,21,21,32
Kenya,30,24,29,26,23
Ghana,42,41,32,27,32
Tanzania,25,24,24,22,33


In [109]:
#output
data.to_csv("../data/sites_data.csv",index=False)

clinical_profiles.to_csv("../data/clinical_site_profiles.csv",index=False)


In [110]:
risks = pd.read_csv("../data/risks.csv")

print("Sites in operational data:", data["site_id"].nunique())
print("Sites in risk data:", risks["site_id"].nunique())

print(
    "Sites missing from risk data:",
    len(set(data["site_id"]) - set(risks["site_id"]))
)

print(
    "Sites missing from operational data:",
    len(set(risks["site_id"]) - set(data["site_id"]))
)

Sites in operational data: 500
Sites in risk data: 500
Sites missing from risk data: 0
Sites missing from operational data: 0
